# Multi-asset walk-forward robustness

This is a controlled version of the existing rolling walk-forward test. It uses the same common calendar and exact outer fold dates for Dukascopy and Oanda, retains synthetic Saturdays by default, reports every monthly fold, and separates a final holdout period from the development period.

It is not a parameter-search notebook: choose the configuration before looking at the holdout output.

In [1]:
import datetime
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import polars as pl


In [2]:
# Freeze this cell before running the experiment.
CCYS = ['jpy', 'chf', 'cad', 'aud', 'nzd', 'eur', 'gbp']
MODEL_LOOKBACKS = [1, 7, 14, 21, 28, 35, 42, 49, 98, 147, 196]
REMOVE_SYNTHETIC_SATURDAY = False

# Match the existing approximately-five-year / one-calendar-month walk-forward design.
TRAIN_DAYS = 30 * 12 * 5
TEST_DAYS = 30
SEED = 15684

COMMON_START = datetime.datetime(2016, 1, 3, tzinfo=datetime.timezone.utc)
COMMON_END = datetime.datetime(2026, 8, 31, tzinfo=datetime.timezone.utc)
# Only use development folds to change the specification. Holdout folds are reporting-only.
FINAL_HOLDOUT_START = datetime.datetime(2025, 1, 1, tzinfo=datetime.timezone.utc)

MODEL_PARAMS = {
    'n_estimators': 350, 'learning_rate': 0.03, 'max_depth': 4,
    'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.7,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'early_stopping_rounds': 30,
    'eval_metric': 'rmse', 'tree_method': 'hist', 'random_state': SEED, 'n_jobs': -1,
}


In [3]:
# Import definitions only; no cells that train models or write artifacts are executed.
source_notebook = Path('multi_asset_trading.ipynb')
if not source_notebook.exists():
    source_notebook = Path('src/notebooks/multi_asset_trading.ipynb')
source_cells = json.loads(source_notebook.read_text(encoding='utf-8'))['cells']
shared = {'include_minors': False, 'dukas_data': True}
for index in (0, 3, 4, 5, 7):
    exec(''.join(source_cells[index]['source']), shared)
if shared['CCYS'] != CCYS:
    raise ValueError(f'Currency mismatch: {shared["CCYS"]} != {CCYS}')


In [4]:
def build_raw(source: str) -> pl.DataFrame:
    reader = shared['read_dukas_data'] if source == 'dukas' else shared['read_oanda_data']
    bars = {ccy: reader(ccy) for ccy in CCYS}
    frames = [
        bars[ccy].select(
            'timestamp',
            pl.col('open').alias(f'{ccy}_open'), pl.col('high').alias(f'{ccy}_high'),
            pl.col('low').alias(f'{ccy}_low'), pl.col('close').alias(f'{ccy}_close'),
        )
        for ccy in CCYS
    ]
    return pl.concat(frames, how='align_full').sort('timestamp').drop_nulls()

raw = {'dukas': build_raw('dukas'), 'oanda': build_raw('oanda')}
common_timestamps = raw['dukas'].select('timestamp').join(raw['oanda'].select('timestamp'), on='timestamp')

def prepare(frame: pl.DataFrame) -> pl.DataFrame:
    result = frame.join(common_timestamps, on='timestamp').filter(
        pl.col('timestamp').is_between(COMMON_START, COMMON_END)
    )
    if REMOVE_SYNTHETIC_SATURDAY:
        result = result.filter(pl.col('timestamp').dt.weekday() != 6)
    return shared['add_model_features'](result, MODEL_LOOKBACKS)

data = {source: prepare(frame) for source, frame in raw.items()}
pl.concat([
    frame.group_by(pl.col('timestamp').dt.weekday().alias('weekday')).len().with_columns(pl.lit(source).alias('source'))
    for source, frame in data.items()
]).sort(['source', 'weekday'])


weekday,len,source
i8,u32,str
1,557,"""dukas"""
2,556,"""dukas"""
3,556,"""dukas"""
4,556,"""dukas"""
5,554,"""dukas"""
…,…,…
3,556,"""oanda"""
4,556,"""oanda"""
5,554,"""oanda"""


In [5]:
@dataclass(frozen=True)
class OuterFold:
    number: int
    train_start: datetime.datetime
    train_end: datetime.datetime
    test_start: datetime.datetime
    test_end: datetime.datetime

def outer_folds(start: datetime.datetime, end: datetime.datetime):
    train_start = start
    train_end = train_start + datetime.timedelta(days=TRAIN_DAYS)
    test_start = train_end + datetime.timedelta(days=1)
    test_end = test_start + datetime.timedelta(days=TEST_DAYS)
    number = 0
    while test_end <= end:
        yield OuterFold(number, train_start, train_end, test_start, test_end)
        number += 1
        shift = datetime.timedelta(days=TEST_DAYS + 1)
        train_start, train_end = train_start + shift, train_end + shift
        test_start, test_end = test_start + shift, test_end + shift

folds = list(outer_folds(COMMON_START, COMMON_END))
fold_index = pl.DataFrame([fold.__dict__ for fold in folds])
fold_index


number,train_start,train_end,test_start,test_end
i64,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]"
0,2016-01-03 00:00:00 UTC,2020-12-07 00:00:00 UTC,2020-12-08 00:00:00 UTC,2021-01-07 00:00:00 UTC
1,2016-02-03 00:00:00 UTC,2021-01-07 00:00:00 UTC,2021-01-08 00:00:00 UTC,2021-02-07 00:00:00 UTC
2,2016-03-05 00:00:00 UTC,2021-02-07 00:00:00 UTC,2021-02-08 00:00:00 UTC,2021-03-10 00:00:00 UTC
3,2016-04-05 00:00:00 UTC,2021-03-10 00:00:00 UTC,2021-03-11 00:00:00 UTC,2021-04-10 00:00:00 UTC
4,2016-05-06 00:00:00 UTC,2021-04-10 00:00:00 UTC,2021-04-11 00:00:00 UTC,2021-05-11 00:00:00 UTC
…,…,…,…,…
62,2021-04-08 00:00:00 UTC,2026-03-13 00:00:00 UTC,2026-03-14 00:00:00 UTC,2026-04-13 00:00:00 UTC
63,2021-05-09 00:00:00 UTC,2026-04-13 00:00:00 UTC,2026-04-14 00:00:00 UTC,2026-05-14 00:00:00 UTC
64,2021-06-09 00:00:00 UTC,2026-05-14 00:00:00 UTC,2026-05-15 00:00:00 UTC,2026-06-14 00:00:00 UTC


In [6]:
def score_weights(weights: pl.DataFrame, frame: pl.DataFrame) -> pl.DataFrame:
    labelled = frame
    for ccy in CCYS:
        labelled = shared['calculate_target'](labelled, ccy)
    targets = [f'{ccy}_target' for ccy in CCYS]
    return (
        weights.join(labelled.select(['timestamp', *targets]), on='timestamp')
        .drop_nulls(targets)
        .with_columns(pl.sum_horizontal([
            pl.col(f'{ccy}_weight') * pl.col(f'{ccy}_target') for ccy in CCYS
        ]).alias('portfolio_return'))
        .sort('timestamp')
    )

def return_metrics(frame: pl.DataFrame) -> dict:
    values = frame['portfolio_return'].to_numpy()
    equity = np.prod(1 + values)
    volatility = np.std(values, ddof=1) if len(values) > 1 else 0.0
    return {
        'rows': len(values), 'total_return': float(equity - 1),
        'mean_return_bp': float(np.mean(values) * 1e4),
        'sharpe': None if volatility == 0 else float(np.mean(values) / volatility * np.sqrt(252)),
    }

def fit_fold(train_source: str, fold: OuterFold):
    train = data[train_source].filter(pl.col('timestamp').is_between(fold.train_start, fold.train_end))
    return shared['train_xgboost_models'](train, model_params=MODEL_PARAMS)


In [7]:
# This is intentionally expensive: 7 currency models × outer folds × two training sources.
# Keep the configuration fixed while it runs.
fold_records = []
daily_returns = []
validation_records = []

for fold in folds:
    for train_source in ('dukas', 'oanda'):
        models, validation, feature_columns = fit_fold(train_source, fold)
        validation_records.extend(validation.with_columns(
            pl.lit(fold.number).alias('fold'), pl.lit(train_source).alias('train_source')
        ).to_dicts())
        for execution_source in ('dukas', 'oanda'):
            test = data[execution_source].filter(pl.col('timestamp').is_between(fold.test_start, fold.test_end))
            weights = shared['generate_weights'](models, test, feature_columns)
            realised = score_weights(weights, data[execution_source])
            period = 'holdout' if fold.test_start >= FINAL_HOLDOUT_START else 'development'
            fold_records.append({
                'fold': fold.number, 'train_source': train_source, 'execution_source': execution_source,
                'period': period, 'test_start': fold.test_start, 'test_end': fold.test_end,
                **return_metrics(realised),
            })
            daily_returns.extend(realised.select('timestamp', 'portfolio_return').with_columns(
                pl.lit(fold.number).alias('fold'), pl.lit(train_source).alias('train_source'),
                pl.lit(execution_source).alias('execution_source'), pl.lit(period).alias('period'),
            ).to_dicts())

fold_results = pl.DataFrame(fold_records).sort(['train_source', 'execution_source', 'fold'])
returns_by_day = pl.DataFrame(daily_returns).sort(['train_source', 'execution_source', 'timestamp'])
validation_results = pl.DataFrame(validation_records).sort(['train_source', 'fold', 'ccy'])
fold_results


fold,train_source,execution_source,period,test_start,test_end,rows,total_return,mean_return_bp,sharpe
i64,str,str,str,"datetime[μs, UTC]","datetime[μs, UTC]",i64,f64,f64,f64
0,"""dukas""","""dukas""","""development""",2020-12-08 00:00:00 UTC,2021-01-07 00:00:00 UTC,27,-0.023308,-8.718885,-8.720652
1,"""dukas""","""dukas""","""development""",2021-01-08 00:00:00 UTC,2021-02-07 00:00:00 UTC,31,-0.00132,-0.41311,-0.400769
2,"""dukas""","""dukas""","""development""",2021-02-08 00:00:00 UTC,2021-03-10 00:00:00 UTC,31,-0.010981,-3.524492,-2.030063
3,"""dukas""","""dukas""","""development""",2021-03-11 00:00:00 UTC,2021-04-10 00:00:00 UTC,31,0.011969,3.853576,3.484621
4,"""dukas""","""dukas""","""development""",2021-04-11 00:00:00 UTC,2021-05-11 00:00:00 UTC,31,0.010425,3.35794,3.409184
…,…,…,…,…,…,…,…,…,…
62,"""oanda""","""oanda""","""holdout""",2026-03-14 00:00:00 UTC,2026-04-13 00:00:00 UTC,31,-0.003971,-1.273261,-1.398102
63,"""oanda""","""oanda""","""holdout""",2026-04-14 00:00:00 UTC,2026-05-14 00:00:00 UTC,31,0.000683,0.226909,0.305373
64,"""oanda""","""oanda""","""holdout""",2026-05-15 00:00:00 UTC,2026-06-14 00:00:00 UTC,31,0.010233,3.287951,6.348077


In [8]:
def aggregate_metrics(frame: pl.DataFrame) -> dict:
    values = frame['portfolio_return'].to_numpy()
    equity = np.prod(1 + values)
    volatility = np.std(values, ddof=1)
    return {
        'rows': len(values), 'total_return': float(equity - 1),
        'annualised_return': float(equity ** (252 / len(values)) - 1),
        'annualised_volatility': float(volatility * np.sqrt(252)),
        'sharpe': float(np.mean(values) / volatility * np.sqrt(252)),
        'mean_return_bp': float(np.mean(values) * 1e4),
    }

summary_records = []
for train_source in ('dukas', 'oanda'):
    for execution_source in ('dukas', 'oanda'):
        for period in ('development', 'holdout'):
            subset = returns_by_day.filter(
                (pl.col('train_source') == train_source) &
                (pl.col('execution_source') == execution_source) &
                (pl.col('period') == period)
            )
            if subset.height > 1:
                summary_records.append({
                    'train_source': train_source, 'execution_source': execution_source, 'period': period,
                    **aggregate_metrics(subset),
                })

walk_forward_summary = pl.DataFrame(summary_records).sort(['period', 'train_source', 'execution_source'])
fold_distribution = fold_results.group_by(['period', 'train_source', 'execution_source']).agg(
    pl.len().alias('folds'), pl.mean('mean_return_bp').alias('mean_fold_daily_bp'),
    pl.median('mean_return_bp').alias('median_fold_daily_bp'),
    (pl.col('mean_return_bp') > 0).mean().alias('positive_fold_fraction'),
    pl.std('mean_return_bp').alias('fold_daily_bp_std'),
).sort(['period', 'train_source', 'execution_source'])

walk_forward_summary, fold_distribution


(shape: (8, 9)
 ┌────────────┬────────────┬────────────┬──────┬───┬────────────┬────────────┬──────────┬───────────┐
 │ train_sour ┆ execution_ ┆ period     ┆ rows ┆ … ┆ annualised ┆ annualised ┆ sharpe   ┆ mean_retu │
 │ ce         ┆ source     ┆ ---        ┆ ---  ┆   ┆ _return    ┆ _volatilit ┆ ---      ┆ rn_bp     │
 │ ---        ┆ ---        ┆ str        ┆ i64  ┆   ┆ ---        ┆ y          ┆ f64      ┆ ---       │
 │ str        ┆ str        ┆            ┆      ┆   ┆ f64        ┆ ---        ┆          ┆ f64       │
 │            ┆            ┆            ┆      ┆   ┆            ┆ f64        ┆          ┆           │
 ╞════════════╪════════════╪════════════╪══════╪═══╪════════════╪════════════╪══════════╪═══════════╡
 │ dukas      ┆ dukas      ┆ developmen ┆ 1478 ┆ … ┆ 0.043829   ┆ 0.032228   ┆ 1.347189 ┆ 1.722917  │
 │            ┆            ┆ t          ┆      ┆   ┆            ┆            ┆          ┆           │
 │ dukas      ┆ oanda      ┆ developmen ┆ 1478 ┆ … ┆ 0.019547   ┆ 0

## Decision rule

A result is more credible when native and source-transfer results have positive development and holdout Sharpe, a majority of monthly folds are positive, and the holdout is not materially worse than development.

Do not choose a specification because it has the highest holdout Sharpe. If you revise features or parameters, move the holdout boundary later or treat the old holdout as development.